# UCLA SWE gold for ST-GNN HUCs

This notebook drives **batch creation of UCLA / NSIDC snow reanalysis “gold” CSVs** for every HUC that appears in an ST-GNN model-results export.

- **Source**: `get_ucla_gold_stgnn` downloads WUS_UCLA_SR tiles, clips to each HUC, and uploads `ucla_swe_in_{huc_id}.csv` to the `snowml-gold` S3 bucket.
- **Prerequisites**: NASA Earthdata credentials (environment variables, `~/.netrc`, or constants in `get_ucla_gold_stgnn.py`). NSIDC access and GES DISC app authorization may be required.
- **Cost / time**: One row per HUC × years in range; each HUC triggers many NetCDF downloads—plan runtime and NSIDC load accordingly.

In [ ]:
# Batch UCLA SWE gold extraction for many HUCs (NSIDC → clip → S3 snowml-gold).
# Same folder as get_ucla_gold_stgnn.py — use a kernel cwd of this directory if the import fails.
import pandas as pd
from get_ucla_gold_stgnn import get_gold_multi

## Model results → HUC list

The CSV is expected to contain a `huc_id` column (e.g. 12-digit HUC12 strings). Unique IDs are passed to `get_gold_multi`.

In [9]:
# ST-GNN evaluation export: must include huc_id for the HUC list below.
MODEL_OUTPUT_CSV = "st_gnn_time_split_test_results_all_runs.csv"
df = pd.read_csv(MODEL_OUTPUT_CSV)

In [ ]:
# One gold file per distinct HUC (total 626) 
# Large lists mean long runtimes and many NSIDC requests.
huc_ids = df["huc_id"].dropna().unique().tolist()
len(huc_ids)

626

In [ ]:
# Water year WY year_start starts Oct 1 of that calendar year; loop is range(year_start, year_end).
year_start = 2009  # first water year included (Oct 1, 2009)
year_end = 2021  # exclusive → last full WY ends Sep 30, 2021

overwrite = True  # False: skip HUC if ucla_swe_in_{huc}.csv already exists in snowml-gold
keep_all_stats = True  # True: SWE_Post_mean/std/median/p25/p75; False: mean column only

error_years = get_gold_multi(
    huc_ids, year_start, year_end, overwrite=overwrite, keep_all_stats=keep_all_stats
)
error_years